In [1]:
import math

import phe.encoding
from phe import paillier

import os
path_project = os.path.dirname(os.path.abspath('.'))
import sys
sys.path.append(os.path.join(path_project, 'src'))
sys.path.append(os.path.join(path_project, 'exp/script'))
import models
from collections import OrderedDict
import numpy as np
import torch

In [10]:
class ExampleEncodedNumber(phe.encoding.EncodedNumber):
    BASE = 64
    LOG2_BASE = math.log(BASE, 2)


def integerize(val: np.array, constant: int=1e16):
    return (val * constant).astype(np.int64)

def weighting_params(model, params: OrderedDict, r: int, n: int):
    weighted_params = OrderedDict()
    for name, param in model.named_parameters():
        if param.requires_grad:
            weighted_params[name] = params[name] * n / r
    return weighted_params

def vector_decrypt(private_key: phe.PaillierPrivateKey, value: np.array, constant: int=1e16):
    return np.vectorize(lambda x: private_key.decrypt_encoded(x, ExampleEncodedNumber).decode())(value) / constant

In [11]:
n_silos = 2
n_users = 4
CONSTANT = 1e16

### CLINET SIDE 1
print("CLIENT SIDE 1")

# data size list for each user and silo
n_list = [
    [10, 40, 30, 10],
    [0, 50, 0, 1]
]

# random mask for each user
r_list = np.random.randint(0, 1e8, size=(n_users,))
print("CLIENT VIEW: r_list", r_list)
print("CLIENT[0] VIEW: ", n_list[0])
print("CLIENT[1] VIEW: ", n_list[1])


### SERVER SIDE 1
print("SERVER SIDE 1")

# summation by users by Secure Aggregation
# Server can see the aggregated data only
N_list = [sum([n_list[j][i] for j in range(n_silos)]) for i in range(n_users)]
print("SERVER VIEW: N_list", N_list)

N_r_list = [N_list[i] / r_list[i] for i in range(n_users)]
print("SERVER VIEW: N_r_list", N_r_list)

inv_N_r_list = [int(1.0 / N_r) for N_r in N_r_list]
print("SERVER VIEW: inv_N_r_list", inv_N_r_list)

print("Generating paillier keypair")
public_key, private_key = paillier.generate_paillier_keypair()

encoded_inv_N_r_list = [ExampleEncodedNumber.encode(public_key, inv_N_r) for inv_N_r in inv_N_r_list]
encrypted_inv_N_r_list = [public_key.encrypt(encoded_inv_N_r) for encoded_inv_N_r in encoded_inv_N_r_list]


### CLIENT SIDE 2 
print("CLIENT SIDE 2")
print("CLIENT VIEW: encrypted_inv_N_r_list", encrypted_inv_N_r_list)

silo_list = []

for i in range(n_silos):
    model_list = [models.create_model('cnn', 'heart_disease', seed=0) for j in range(n_users)]
    params_list = [model_list[j].state_dict() for j in range(n_users)]
    weighted_params_list = [
        weighting_params(model_list[0], params_list[j], r_list[j], n_list[i][j])
        for j in range(n_users)
    ]
    for s, (weight_params, encrypted_inv_N_r) in enumerate(zip(weighted_params_list, encrypted_inv_N_r_list)):
        if s == 0:
            sum_weight_params = weight_params
            for key, value in weight_params.items():
                sum_weight_params[key] = integerize(value.numpy(), CONSTANT) * encrypted_inv_N_r
        else:
            for key, value in weight_params.items():
                sum_weight_params[key] += integerize(value.numpy(), CONSTANT) * encrypted_inv_N_r
    silo_list.append(sum_weight_params)


### SERVER SIDE 2
print("SERVER SIDE 2")

global_model = models.create_model('cnn', 'heart_disease', seed=0)
global_params = global_model.state_dict()
print(global_params['linear.weight'][0][0].item())

for s in range(n_silos):
    if s == 0:
        global_params = silo_list[s]
    else:
        for key, value in silo_list[s].items():
            global_params[key] += value

for key, value in global_params.items():
    global_params[key] = torch.from_numpy(vector_decrypt(private_key, value, CONSTANT)) / n_users

print(global_params['linear.weight'][0][0].item())

CLIENT SIDE 1
CLIENT VIEW: r_list [64036100 26864998 83282290 11692323]
CLIENT[0] VIEW:  [10, 40, 30, 10]
CLIENT[1] VIEW:  [0, 50, 0, 1]
SERVER SIDE 1
SERVER VIEW: N_list [10, 90, 30, 11]
SERVER VIEW: N_r_list [1.5616191491986552e-07, 3.35008400149518e-06, 3.602206423478509e-07, 9.40788241994341e-07]
SERVER VIEW: inv_N_r_list [6403610, 298499, 2776076, 1062938]
Generating paillier keypair
CLIENT SIDE 2
CLIENT VIEW: encrypted_inv_N_r_list [<phe.paillier.EncryptedNumber object at 0x2a03d5f40>, <phe.paillier.EncryptedNumber object at 0x2a03d5e20>, <phe.paillier.EncryptedNumber object at 0x13fda1460>, <phe.paillier.EncryptedNumber object at 0x13fd931c0>]
SERVER SIDE 2
-0.002076476812362671
-0.0020764748126338
